# Homework 2 - Class Activation Maps and Adversarial Examples

## Part 1. Class Activation Map with convolutions

In the first part, we will generate class activation maps as described in the paper [Learning Deep Features for Discriminative Localization (Zhou et al., 2016)](https://arxiv.org/pdf/1512.04150). Note that there is a [GitHub repo](https://github.com/zhoubolei/CAM) associated with the paper, and even a [demo in PyTorch](https://github.com/zhoubolei/CAM/blob/master/pytorch_CAM.py). The code in this part is adapted from this demo.

Class Activation Mapping (CAM) is a technique for visualizing which regions of an input image a convolutional neural network (CNN) used to predict a particular class. This allows us to interpret the outputs of CNNs. The output is a heatmap overlaid on the image, highlighting the areas most relevant to the decision. You can see illustrative examples in Figure 1 of the paper.

In [ ]:
import io
import requests
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.nn import functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import cv2

### Helper functions for plotting

In [ ]:
def pil_2_np(img_pil):
    # transform a PIL image in a numpy array
    return np.asarray(img_pil)

def display_np(img_np):
    plt.imshow(Image.fromarray(np.uint8(img_np)))
    plt.axis('off')
    plt.show()
    
def plot_CAM(img_np, CAM):
    height, width, _ = img_np.shape
    CAM = 255 - CAM        # Red means higher values, blue means lower values
    heatmap = cv2.applyColorMap(cv2.resize(CAM,(width, height)), cv2.COLORMAP_JET)
    result = heatmap * 0.8 + img_np * 0.2
    display_np(result)

def plot_img_tensor(img):
    plt.imshow(np.transpose(img.detach().numpy(), [1,2,0]))
    plt.axis('off')
    plt.show()

def plot_untransform(x_t : torch.Tensor): 
    t_std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
    t_mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
    x_np = (x_t * t_std + t_mean).detach().numpy()
    x_np = np.transpose(x_np, [1, 2, 0])
    plt.imshow(x_np)
    plt.axis('off')
    plt.show()

### Loading ImageNet labels and a sample image

In [ ]:
LABELS_URL = 'https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json'
IMG_URL = 'https://raw.githubusercontent.com/zhoubolei/CAM/master/test.jpg'

We define the `idx_to_label` and `label_to_idx` dictionaries to map the ImageNet labels to indices and vice versa.

In [ ]:
labels_json = requests.get(LABELS_URL).json()
idx_to_label = {int(k): v[1] for k, v in labels_json.items()}
label_to_idx = {v[1]: int(k) for k, v in labels_json.items()}

We will generate a CAM on the following ImageNet picture.

In [ ]:
response = requests.get(IMG_URL)
response.raise_for_status()
img = Image.open(io.BytesIO(response.content))
img

We preprocess it to obtain a PyTorch tensor `input_image` ready to use.

In [ ]:
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   normalize
])

input_image = preprocess(img).unsqueeze(0)
input_image.shape

### Extracting CAMs from ResNet18

As in the demo, we will use the Resnet18 architecture. In order to get CAM, we need to transform this network in a fully convolutional network: at all layers, we need to deal with images, i.e. with a shape $\text{Number of channels} \times W\times H$ . In particular, we are interested in the last images. As we deal with a Resnet18 architecture, the image obtained before applying the `AdaptiveAvgPool2d` has size 1 \times $512\times 7 \times 7$ if the input has size $1\times 3\times 224\times 224 $:

![resnet_Archi](https://pytorch.org/assets/images/resnet.png)

In [ ]:
net = models.resnet18(pretrained=True)
net.eval()

A - First, remove the last layers of the resnet18 model which are called `(avgpool)` and `(fc)`. Check that for an original image of size $1\times 3\times 224\times 224 $, you obtain an image of size $1 \times 512\times 7\times 7$.

In [ ]:
# TODO : Create a net_removed model by removing the last layers

In [ ]:
# TODO : Check output size when giving input_image as an input to net_removed

B - Retrieve the weights (and bias) of the `fc` layer, i.e. a matrix of size $1000\times 512$ transforming a vector of size 512 into a vector of size 1000 to make the prediction. Then, use these weights and bias to apply it pixelwise in order to transform your $1 \times 512\times 7\times 7$ image into a $1 \times 1000\times 7\times 7$ output. You can interpret this output as follows: `output[0,i,j,k]` is the logit for 'pixel' `[j,k]` for being of class `i`.

Hint: use `F.conv2d`.

In [ ]:
# TODO : Retrieve the weights and biases of the last fully connected layer of the original network. Name them weights and biases respectively.

In [ ]:
# TODO : Create a function that applies the weights and biases pixelwise to the output of the reduced network

def apply_fc_to_conv_output(conv_output, weights_fc, biases_fc):
    return 

In [ ]:
# Testing the function: output size should be (1, 1000, 7, 7)

conv_output = output_image  # Output from net_removed
fc_output = apply_fc_to_conv_output(conv_output, weights, biases)
fc_output.shape # Should be (1, 1000, 7, 7)

C - From this $1 \times 1000\times 7\times 7$ output, check that you can retrieve the original output class (i.e. ```mountain_bike```) given by the `resnet18` by using `F.adaptive_avg_pool2d`.

In [ ]:
# TODO : Get the predicted class for the input image

Explain why this gives the same prediction as the original ResNet18.

<span style="color:red">Replace by your answer</span>

D - We can now plot the CAMs, which are simply heatmaps obtained from the $1\times 1000\times 7\times 7$ output. Each CAM corresponds to a label $i \in \{0,\dots,999\}$. First, write a `return_CAM` function that takes as input a $1\times 1000\times W\times H$ tensor and a label in $\{0,\dots,999\}$, and returns a NumPy array of size $W \times H$ with integer entries between $0$ and $255$ for the display. Then, using the `plot_CAM` helper function defined at the beginning of this part, plot the CAMs corresponding to the classes `mountain_bike` and `lakeside`.

In [ ]:
# TODO : return_CAM function

def returnCAM(feature_conv, idx):
    # input: tensor feature_conv of dim 1000*W*H and idx between 0 and 999
    # output: image W*H with scaled integer coefficients between 0 and 255 for the display
    return

In [ ]:
# TODO : Plot the CAM for the class mountain_bike over the original image

In [ ]:
# TODO : Plot the CAM for the class lakeside over the original image

Is the reduced (fully convolutional) network working with input images of size different from $224 \times 224$ ? What about ResNet18 ? Explain why.

<span style="color:red">Replace by your answer</span>

##  Part 2. Adversarial examples

In this second part, we will look at [adversarial examples](https://arxiv.org/abs/1607.02533): "An adversarial example is a sample of input data which has been modified very slightly in a way that is intended to cause a machine learning classifier to misclassify it. In many cases, these modifications can be so subtle that a human observer does not even notice the modification at all, yet the classifier still makes a mistake. Adversarial examples pose security concerns because they could be used to perform an attack on machine learning systems..."

Rules of the game:
- the attacker cannot modify the classifier, i.e. the neural net with the preprocessing done on the image before being fed to the network. 
- even if the attacker cannot modify the classifier, we assume that the attacker knows the architecture of the classifier. Here, we will still work with `resnet18` and the standard Imagenet normalization. 
- the attacker can only modify the physical image fed into the network.
- the attacker should fool the classifier, i.e. the label obtained on the corrupted image should not be the same as the label predicted on the original image.

First, you will implement *Fast gradient sign method (FGSM)* which is described in Section 2.1 of [Adversarial examples in the physical world](https://arxiv.org/abs/1607.02533). The idea is simple, suppose you have an image $\mathbf{x}$ and when you pass it through the network, you get the 'true' label $y$. You know that your network has been trained by minimizing the loss $J(\mathbf{\theta}, \mathbf{x}, y)$ with respect to the parameters of the network $\theta$. Now, $\theta$ is fixed as you cannot modify the classifier so you need to modify $\mathbf{x}$. In order to do so, you can compute the gradient of the loss with respect to $\mathbf{x}$ i.e. $\nabla_{\mathbf{x}} J(\mathbf{\theta}, \mathbf{x}, y)$ and use it as follows to get the modified image $\tilde{\mathbf{x}}$:
$$
\tilde{\mathbf{x}} = \text{Clamp}\left(\mathbf{x} + \epsilon *
\text{sign}(\nabla_{\mathbf{x}} J(\mathbf{\theta}, \mathbf{x}, y)),\mathbf{a},\mathbf{b}\right),
$$
where $\text{Clamp}(\cdot, \mathbf{a},\mathbf{b})$ clips the values so that $\tilde{\mathbf{x}}$ is a proper image.
Note that if instead of sign, you take the full gradient, you are now following the gradient i.e. increasing the loss $J(\mathbf{\theta}, \mathbf{x}, y)$ so that $y$ becomes less likely to be the predicted label.

This is the image we will be working on:

In [ ]:
url_car = 'https://cdn130.picsart.com/263132982003202.jpg?type=webp&to=min&r=640'
response = requests.get(url_car)
img_pil = Image.open(io.BytesIO(response.content))

In [ ]:
plt.imshow(img_pil)
plt.axis('off')
plt.show()

We do not need to compute gradients with respect to ResNet18's parameters:

In [ ]:
for p in net.parameters():
    p.requires_grad = False

Just like previsouly, we preprocess the image to obtain a PyTorch tensor we can work with.

In [ ]:
preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   normalize
])

x = preprocess(img_pil).clone().unsqueeze(0)

We can check that ResNet18 predicts the correct label for this image.

In [ ]:
logit = net(x)
y = logit.argmax().item()
idx_to_label[y]

This function will be convenient to obtain predictions from the logits outputted by the ResNet18.

In [ ]:
def print_preds(scores):
    # print the predictions with their 'probabilities' from the scores
    h_x = F.softmax(scores, dim=1).data.squeeze()
    probs, idx = h_x.sort(0, True)
    probs = probs.numpy()
    idx = idx.numpy()
    # output the prediction
    for i in range(0, 5):
        print('{:.3f} -> {}'.format(probs[i], idx_to_label[idx[i]]))
    return idx

1. Implement this attack, and display the corrupted image (use the `plot_untransform` helper function for plotting). Keep in mind that ideally, the corrupted image should be barely distinguishable from the original. 

Hint: to know what is the correct values $\mathbf{a},\mathbf{b}$ for clipping, keep in mind that the original images had coefficients in $[0,1]$, and that we applied the normalization $(x-t_\mathrm{mean})/t_\mathrm{std}$, where $t_{\mathrm{mean}}, t_{\mathrm{{std}}}$ are defined in the cell below.

In [ ]:
# TODO : Implement the fgsm_attack function

t_std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
t_mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)

def fgsm_attack(image, epsilon, data_grad):
    # Collect the element-wise sign of the data gradient

    # Create the perturbed image by adjusting each pixel of the input image

    # Clip the coefficients to keep the values in an appropriate range

    # Return the perturbed image
    return 

In [ ]:
# TODO : Compute the appropriate gradient and use fgsm_attack 

In [ ]:
# TODO : plot the image using plot_untransform

2. For what value of epsilon is your attack successful? What is the predicted class then?

<span style="color:red">Replace by your answer</span>


In [ ]:
# TODO : Predict the class of the adversarial image with the original network

3. The gradient is the same size as the image. Print as an image the signs of the coefficients of this gradient (use `plot_img_tensor`), and pass this "sign tensor" through the network. What prediction do you obtain? Compare to [Explaining and Harnessing Adversarial Examples](https://arxiv.org/abs/1412.6572) 

<span style="color:red">Replace by your answer</span>

In [ ]:
# TODO : Print the signs of the gradient as an image

In [ ]:
# TODO : compute the prediction of ResNet18 obtained when giving the gradient signs as input

## Part 3. Transforming a car into a cat

We now implement the *Iterative Target Class Method (ITCM)* as defined by equation (4) in [Adversarial Attacks and Defences Competition](https://arxiv.org/abs/1804.00097). The objective here is to take a starting image $\tilde{\mathbf{x}}_{0}$ with class $y_0$, and to modify it to make the model (of parameters $\theta$) predict another class $y\neq y_0$. The idea here is to iteratively decrease the loss of the image $\tilde{\mathbf{x}}_t$ on the class $y$:
\begin{align*}
    \tilde{\mathbf{x}}_{t+1} = \mathrm{Clamp}(\tilde{\mathbf{x}}_t - \gamma \nabla_\mathbf{x} J (\theta,\tilde{\mathbf{x}}_t,y),\mathbf{a},\mathbf{b})
\end{align*}

To test it, we will transform the car (labeled minivan by our `resnet18`) into a [Tabby cat](https://en.wikipedia.org/wiki/Tabby_cat) (class 281 in Imagenet). But you can try with any other target.

1. Implement the ITCM and display the resulting image. 

In [ ]:
# xd will me the imagie to modify, on which ITCM will be applied.

x = preprocess(img_pil).clone()
xd = preprocess(img_pil).clone()
xd.requires_grad = True

In [ ]:
idx = 281 #tabby
optimizer = optim.SGD([xd], lr=0.01)

for i in range(200):
    #TODO: your code here

    # TODO: break the loop once we are satisfied
    if ?:
        break

In [ ]:
_ = print_preds(output)

In [ ]:
# TODO : plot the corrupted image

## Part 4. Where is the cat hidden?

Last, we use CAM on our newly generated adversarial example to understand where the network see a cat in the image.

1. Display the CAM for the class tabby

In [ ]:
# TODO : display the CAM for the class tabby

2. Display the CAM for the class minivan

In [ ]:
# TODO : display the CAM for the class minivan